In [82]:
%load_ext autoreload
%autoreload 2

import os
import re
import sys
import time
import clingo
import joblib
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from utils.data_utils import extract_from_json, aggregate_to_operator_day
from utils.asp_utils import generate_clingo_facts, generate_baseline_facts, generate_ml_facts
from utils.model_utils import TabPFNQuantileWrapper, TARGET_COLS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [83]:
try:
    train_columns = joblib.load("saved_models/train_columns.pkl")
    
    models_single = {}
    path_single = "saved_models/tabpfn_model_target_assignments.pkl"
    if os.path.exists(path_single):
        models_single['target_assignments'] = joblib.load(path_single)
    
    models_multi = {}
    multi_targets = ['target_assN', 'target_assO', 'target_assA', 'target_assCP', 'target_assCN', 'target_assMAC']
    for t in multi_targets:
        path_multi = f"saved_models/tabpfn_model_{t}.pkl"
        if os.path.exists(path_multi):
            models_multi[t] = joblib.load(path_multi)
            
    print(f"Modelli Single-Target caricati: {len(models_single)}")
    print(f"Modelli Multi-Target caricati: {len(models_multi)}")
except FileNotFoundError:
    print("Errore: Modelli o file delle colonne non trovati.")

Modelli Single-Target caricati: 1
Modelli Multi-Target caricati: 6


In [84]:
def parse_assignments_to_df(raw_assignments):
    """
    Prende la lista di stringhe di Clingo e la trasforma in un DataFrame
    con colonne ['Patient', 'Operator']
    """
    parsed_data = []
    # Cerca la parola assignment, poi cattura il primo numero (Operatore) e il secondo (Paziente)
    pattern = re.compile(r"assignment\((-?\d+),\s*(\d+)")
    
    for assignment in raw_assignments:
        match = pattern.search(assignment)
        if match:
            op_id = int(match.group(1))
            pat_id = int(match.group(2))
            parsed_data.append({"Patient": pat_id, "Operator": op_id})
            
    return pd.DataFrame(parsed_data, columns=["Patient", "Operator"])

In [85]:
def analyze_agenda_shifts(raw_base, raw_ns):
    """
    Confronta gli assegnamenti e restituisce un DataFrame con l'analisi degli scostamenti.
    """
    # 1. Parsiamo le due liste
    df_base = parse_assignments_to_df(raw_base)
    df_ns = parse_assignments_to_df(raw_ns)
    
    # 2. Uniamo i dati basandoci sull'ID del Paziente
    df_compare = pd.merge(df_base, df_ns, on="Patient", suffixes=('_Base', '_ML'), how="outer")
    df_compare = df_compare.fillna(-999).astype(int) # -999 per eventuali dati mancanti
    
    # 3. Classifichiamo cosa è successo ad ogni paziente
    def categorize_shift(row):
        op_base = row['Operator_Base']
        op_ns = row['Operator_ML']
        
        if op_base == op_ns:
            if op_base == -1:
                return "Non Assegnato in Entrambi"
            return "Invariato"
        elif op_base != -1 and op_ns == -1:
            return "Perso dal ML"
        elif op_base == -1 and op_ns != -1:
            return "Salvato dal ML"
        else:
            return "Spostato su nuovo Operatore"

    df_compare['Status'] = df_compare.apply(categorize_shift, axis=1)
    df_compare = df_compare[['Patient', 'Status', 'Operator_Base', 'Operator_ML']]
    
    return df_compare

In [86]:
def get_shift_counts(df_analisi):
    """Estrae i conteggi degli scostamenti in modo sicuro."""
    if df_analisi.empty:
        return 0, 0, 0, 0
    counts = df_analisi['Status'].value_counts()
    invariati = counts.get("Invariato", 0)
    spostati = counts.get("Spostato su nuovo Operatore", 0)
    salvati = counts.get("Salvato dal ML (Tolto da -1)", 0)
    persi = counts.get("Perso dal ML (Aggiunto a -1)", 0)
    return invariati, spostati, salvati, persi

In [87]:
def safe_cost(cost_list, level=1):
    """Estrae il costo in modo sicuro (level 1 = Weak constraint priorità 2)"""
    if cost_list is None or len(cost_list) <= level:
        return 0
    return cost_list[level]

In [88]:
def generate_predictions_for_date(json_path, target_date, train_columns, trained_models, output_filename):
    df_raw = extract_from_json([json_path])
    df_agg = aggregate_to_operator_day(df_raw)
    
    df_today = df_agg[df_agg['planning_date'].astype(str).str.contains(target_date)].copy()
    
    if df_today.empty:
        print("Nessun dato trovato per questa data.")
        return
        
    today_ids = df_today['operator_id']
    X_today = df_today.drop(columns=['operator_id', 'planning_date', 'target_assignments'], errors='ignore')
    
    cat_cols = X_today.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    X_today = pd.get_dummies(X_today, columns=cat_cols, drop_first=True, dtype=int)

    X_today = X_today.reindex(columns=train_columns, fill_value=0)    
    X_today_np = X_today.values.astype(np.float32)
    
    predictions_dict = {}
    for target_name, models_dict in trained_models.items():
        q10 = models_dict['q10'].predict(X_today_np)
        q50 = models_dict['q50'].predict(X_today_np)
        q90 = models_dict['q90'].predict(X_today_np)
        predictions_dict[target_name] = {'q10': q10, 'q50': q50, 'q90': q90}
    
    generate_clingo_facts(
        X_test=df_today,
        predictions_dict=predictions_dict,
        original_ids=today_ids, 
        output_filename=output_filename
    )

In [89]:
def run_neuro_symbolic_solver(rules_files_list, facts_file, ml_file, timeout_seconds=30.0):    
    ctl = clingo.Control(["--opt-strategy=usc,k,0,4", "--opt-usc-shrink=bin"])
    
    for r_file in rules_files_list:
        ctl.load(r_file)
        
    ctl.load(facts_file)
    if ml_file:
        ctl.load(ml_file)
    
    ctl.ground([("base", [])])
    
    best_cost = None
    assignments = []
    unassigned_count = 0
    
    def on_model(model):
        nonlocal best_cost, assignments, unassigned_count
        best_cost = model.cost
        assignments = [str(sym) for sym in model.symbols(shown=True)]
        unassigned_count = sum(1 for a in assignments if "assignment(-1" in a)
    
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout_seconds)
        if not finished:
            handle.cancel()
    
    return assignments, best_cost, unassigned_count

In [ ]:
def run_comprehensive_benchmark(json_path, start_date, end_date, train_cols, m_single, m_multi, timeout=30.0):
    dates = pd.date_range(start=start_date, end=end_date)
    results = []
    
    os.makedirs("results", exist_ok=True)
    os.makedirs("encodings/facts", exist_ok=True)
    
    for date_obj in dates:
        date_str = date_obj.strftime('%Y-%m-%d')
        
        # 1. Generazione Fatti
        try:
            # A. Fatti pesanti per la Baseline
            generate_baseline_facts(json_path, date_str, "encodings/facts/baseline_facts.lp")
            
            # Controllo ospedale vuoto
            with open("encodings/facts/baseline_facts.lp", 'r') as f:
                if len(f.readlines()) < 10:
                    continue
                    
            # B. Fatti snelli e relazionali per il ML
            generate_ml_facts(json_path, date_str, "encodings/facts/ml_physical_facts.lp")
            
            # C. Fatti predittivi TabPFN
            generate_predictions_for_date(json_path, date_str, train_cols, m_single, "encodings/facts/ml_preds_single.lp")
            generate_predictions_for_date(json_path, date_str, train_cols, m_multi, "encodings/facts/ml_preds_multi.lp")
        except Exception as e:
            print(f"[{date_str}] Saltato: Errore generazione fatti -> {e}")
            continue
            
        # --- ROUND 1: BASELINE ---
        t0 = time.perf_counter()
        ass_base, cost_base, unass_base = run_neuro_symbolic_solver(["encodings/base_rules.lp"], "encodings/facts/baseline_facts.lp", None, timeout)
        t_base = time.perf_counter() - t0
        
        # --- ROUND 2: ML SINGLE-TARGET ---
        t0 = time.perf_counter()
        ass_single, cost_single, unass_single = run_neuro_symbolic_solver(["encodings/ml_single_board_rules.lp"], "encodings/facts/ml_physical_facts.lp", "encodings/facts/ml_preds_single.lp", timeout)
        t_single = time.perf_counter() - t0
        
        # --- ROUND 3: ML MULTI-TARGET ---
        t0 = time.perf_counter()
        ass_multi, cost_multi, unass_multi = run_neuro_symbolic_solver(["encodings/ml_multi_board_rules.lp"], "encodings/facts/ml_physical_facts.lp", "encodings/facts/ml_preds_multi.lp", timeout)
        t_multi = time.perf_counter() - t0
        
        # --- ANALISI CLINICA (Rispetto alla Baseline) ---
        df_shifts_single = analyze_agenda_shifts(ass_base, ass_single)
        inv_S, spo_S, sal_S, per_S = get_shift_counts(df_shifts_single)
        
        df_shifts_multi = analyze_agenda_shifts(ass_base, ass_multi)
        inv_M, spo_M, sal_M, per_M = get_shift_counts(df_shifts_multi)
        
        # --- SALVATAGGIO METRICHE ---
        results.append({
            'Data': date_str,
            'Tempo_Base_sec': round(t_base, 4),
            'Tempo_Single_sec': round(t_single, 4),
            'Tempo_Multi_sec': round(t_multi, 4),
            
            'Pazienti_Base': len(ass_base) - unass_base,
            'Pazienti_Single': len(ass_single) - unass_single,
            'Pazienti_Multi': len(ass_multi) - unass_multi,
            
            'Terra_Base': unass_base,
            'Terra_Single': unass_single,
            'Terra_Multi': unass_multi,
            
            'Single_Invariati': inv_S,
            'Single_Spostati': spo_S,
            'Single_Salvati': sal_S,
            'Single_Persi': per_S,
            
            'Multi_Invariati': inv_M,
            'Multi_Spostati': spo_M,
            'Multi_Salvati': sal_M,
            'Multi_Persi': per_M,
            
            'Costo_Base': safe_cost(cost_base, 1),
            # Salviamo il costo di under/over booking (Livelli 3 e 2)
            'Costo_Single_Over': safe_cost(cost_single, 3),
            'Costo_Single_Under': safe_cost(cost_single, 2),
            'Costo_Multi_Over': safe_cost(cost_multi, 3),
            'Costo_Multi_Under': safe_cost(cost_multi, 2)
        })
        
        print(f"[{date_str}] Tempi -> Base: {t_base:.2f}s | Sngl: {t_single:.3f}s | Mlti: {t_multi:.3f}s -- Persi: {per_S}(S) / {per_M}(M)")

    df_results = pd.DataFrame(results)
    csv_path = "results/master_benchmark_results.csv"
    df_results.to_csv(csv_path, index=False)
    
    print("REPORT FINALE: BASELINE vs SINGLE-TARGET vs MULTI-TARGET")
    print(f"TEMPI MEDI DI ESECUZIONE:")
    print(f" - Baseline        : {df_results['Tempo_Base_sec'].mean():.4f}s (Max: {df_results['Tempo_Base_sec'].max():.2f}s)")
    print(f" - ML Single-Target: {df_results['Tempo_Single_sec'].mean():.4f}s (Max: {df_results['Tempo_Single_sec'].max():.2f}s)")
    print(f" - ML Multi-Target : {df_results['Tempo_Multi_sec'].mean():.4f}s (Max: {df_results['Tempo_Multi_sec'].max():.2f}s)\n")
    
    print(f"IMPATTO CLINICO MENSILE (Rispetto alla Baseline):")
    print(f" - Single-Target -> Salvati dal ML: {df_results['Single_Salvati'].sum()} | Persi: {df_results['Single_Persi'].sum()} | Spostati (Bilanciati): {df_results['Single_Spostati'].sum()}")
    print(f" - Multi-Target  -> Salvati dal ML: {df_results['Multi_Salvati'].sum()} | Persi: {df_results['Multi_Persi'].sum()} | Spostati (Bilanciati): {df_results['Multi_Spostati'].sum()}")
    
    return df_results

In [91]:
data_url = 'data/anon_boardHistory_Org6_2023-01-01_2023-01-31.json' 

df_master = run_comprehensive_benchmark(
    json_path=data_url, 
    start_date='2023-01-01', 
    end_date='2023-01-31', 
    train_cols=train_columns,
    m_single=models_single,
    m_multi=models_multi,
    timeout=30.0
)

KeyboardInterrupt: 